# 03 · Data cleaning: rural share (Census 2020 Urban and Rural)

**Goal:** the **% of each Minnesota county's population living in rural areas**, which Q1 names directly ("the rural share") and Q5 Part C uses.

**Source:** US Census Bureau, *2020 Census Urban and Rural*, county file `2020_UA_COUNTY.xlsx`
(https://www2.census.gov/geo/docs/reference/ua/2020_UA_COUNTY.xlsx), saved unedited in `Data/`.
- **Geographic grain:** county (built up from census blocks: each block is classified urban or rural).
- **Time grain:** one-time, **2020**. The Census Bureau only redraws urban/rural areas once a decade (2010 → 2020 → 2030), so 2020 is the latest official version and covers our whole period.
- **Why this over USDA's metro/rural code (`rucc_2023`, already in the MMG file):** it's a **percentage**, not a category. A "metro" county can still have many rural residents (e.g. Stearns).

**Output:** `Data/processed/rural_share_2020_clean.csv` (87 counties)

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"

raw = pd.read_excel(DATA / "2020_UA_COUNTY.xlsx", sheet_name="2020_UA_COUNTY", dtype={"STATE": str, "COUNTY": str})
mmg = pd.read_csv(DATA / "processed" / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
print("all US counties:", raw.shape)

all US counties: (3234, 31)


## Step 1: Keep Minnesota, build FIPS, keep useful columns

- **What:** keep rows with `STATE == "27"` (Minnesota); FIPS = `STATE` + `COUNTY` (e.g. `27` + `001` = `27001`); keep population (total / urban / rural), **% rural**, land area and population density. Convert `POPPCT_RUR` from a fraction (0–1) to a **percent** (0–100).
- **Why:** FIPS is our join key; density helps the access story (people spread out → farther from food shelves).
- **Outcome:** 87 rows, 9 columns, matching the 87 Map the Meal Gap counties exactly.
- **Why not housing-unit or land-area shares:** the question is about *people*; population share is the standard "rural share".

In [2]:
mn = raw[raw["STATE"] == "27"].copy()
mn["fips"] = mn["STATE"] + mn["COUNTY"]

rural = pd.DataFrame({
    "fips": mn["fips"],
    "county": mn["COUNTY_NAME"],
    "pop_2020": mn["POP_COU"].astype(int),
    "pop_urban_2020": mn["POP_URB"].astype(int),
    "pop_rural_2020": mn["POP_RUR"].astype(int),
    "pct_rural_2020": (mn["POPPCT_RUR"].astype(float) * 100).round(1),
    "land_sq_mi": mn["ALAND_Mi²_COU"].astype(float).round(1),
    "pop_per_sq_mi": mn["POPDEN_COU"].astype(float).round(1),
}).sort_values("fips").reset_index(drop=True)

assert len(rural) == 87
assert set(rural["fips"]) == set(mmg["fips"]), "FIPS don't match the MMG county list"
assert (rural["pop_urban_2020"] + rural["pop_rural_2020"] == rural["pop_2020"]).all()
rural["rural_band"] = pd.cut(rural["pct_rural_2020"], [-0.1, 25, 50, 75, 100],
                             labels=["mostly urban (<25%)", "mixed (25-50%)", "mostly rural (50-75%)", "rural (>75%)"])
rural.head()

,fips,county,pop_2020,pop_urban_2020,pop_rural_2020,pct_rural_2020,land_sq_mi,pop_per_sq_mi,rural_band
0,27001,Aitkin,15697,0,15697,100.0,1821.8,8.6,rural (>75%)
1,27003,Anoka,363887,306020,57867,15.9,422.0,862.4,mostly urban (<25%)
2,27005,Becker,35183,10234,24949,70.9,1315.1,26.8,mostly rural (50-75%)
3,27007,Beltrami,46228,14849,31379,67.9,2504.7,18.5,mostly rural (50-75%)
4,27009,Benton,41379,22744,18635,45.0,408.3,101.3,mixed (25-50%)


## Step 2: Checks and save

- **Check:** compare with USDA's metro/rural code (`rucc_2023`): rural-coded counties should mostly have high rural shares, but "metro" counties can still be quite rural.

In [3]:
check = rural.merge(mmg[mmg["year"] == 2024][["fips", "rural", "rucc_2023"]], on="fips", validate="one_to_one")
display(check.groupby("rural")["pct_rural_2020"].describe()[["count", "min", "50%", "max"]].round(1))
print("'metro' counties (USDA) with over 40% rural residents:")
display(check[(~check["rural"]) & (check["pct_rural_2020"] > 40)][["county", "rucc_2023", "pct_rural_2020"]].sort_values("pct_rural_2020", ascending=False))
display(rural["rural_band"].value_counts().sort_index().rename("counties").to_frame())

rural.to_csv(DATA / "processed" / "rural_share_2020_clean.csv", index=False)
print("saved Data/processed/rural_share_2020_clean.csv:", rural.shape)

,count,min,50%,max
rural,,,,
False,27.0,0.0,39.3,100.0
True,60.0,29.7,99.7,100.0


'metro' counties (USDA) with over 40% rural residents:


,county,rucc_2023,pct_rural_2020
22,Fillmore,3.0,100.0
39,Le Sueur,1.0,85.8
47,Mille Lacs,1.0,81.8
78,Wabasha,3.0,81.2
27,Houston,3.0,71.0
12,Chisago,1.0,65.7
8,Carlton,2.0,63.5
19,Dodge,3.0,63.3
29,Isanti,1.0,59.3
70,Sherburne,1.0,50.9


,counties
rural_band,
mostly urban (<25%),8
mixed (25-50%),22
mostly rural (50-75%),20
rural (>75%),37


saved Data/processed/rural_share_2020_clean.csv: (87, 9)
